In [1]:
!pip install langchain

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   --------------------------------------- 548.1/548.1 kB 11.1 MB/s eta 0:00:00

  Attempting uninstall: orjson

    Found existing installation: orjson 3.11.4

    Uninstalling orjson-3.11.4:

      Successfully uninstalled orjson-3.11.4

   ---------------- -----------------------  5/12 [langsmith]
   ---------------- -----------------------  5/12 [langsmith]
   ---------------- -----------------------  5/12 [langsmith]
   -------------------- -------------------  6/12 [langgraph-sdk]
   ----------------------- ----------------  7/12 [langchain-core]
   ----------------------- ----------------  7/12 [langchain-core]
   ----------------------- ----------------  7/12 [langchain-core]
   ----------------------- ----------------  7/12 [langchain-core]
   ----------------------- ----------------  7/12 [langchain-core]
   -------------------------- -------------  8/12 [langgraph-checkpoint]
   --------------------------

In [4]:
!pip install langchain langchain-community langchain-text-splitters

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------- ----------------------- 1.0/2.5 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 6.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 8.1 MB/s eta 0:00:00

  Attempting uninstall: requests

    Found existing installation: requests 2.32.3

    Uninstalling requests-2.32.3:

      Successfully uninstalled requests-2.32.3

   ------------ ---------------------------  3/10 [marshmallow]
  Attempting uninstall: pydantic-settings
   ------------ ---------------------------  3/10 [marshmallow]
    Found existing installation: pydantic-settings 2.6.1
   ------------ ---------------------------  3/10 [marshmallow]
   ------------------------ ---------------  6/10 [pydantic-settings]
    Uninstalling pydantic-settings-2.6.1:
   ------------------------ ---------------  6/10 [pydantic-set

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [17]:
import pandas as pd
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [18]:
DATA_PATH = r"D:\1 Univesrity work\Lect 2\Data semantics\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum\processed\all_subjects_text.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,subject,file_name,text
0,data_management,0-CommonIntro.pdf,data management 2 teaching team 1 course goals...
1,data_management,1-DataLifeCycle.pdf,data lifecycle 2 methodologies 1 tools 2 phase...
2,data_management,11 LLM.pdf,a very general overview llm and data 1 2 large...
3,data_management,2- Data Acquisition.pdf,i need data data acquisition 1 2 data acquisit...
4,data_management,2.0- Data Acquisition.pdf,data management lab 2 data acquisition: web sc...


In [19]:
print(df.shape)

print(df.columns)

(118, 3)
Index(['subject', 'file_name', 'text'], dtype='object')


In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    length_function=len
)

In [21]:
all_chunks = []

In [22]:
for idx, row in df.iterrows():

    text = row["text"]

    subject = row["subject"]

    file_name = row["file_name"]

    chunks = text_splitter.split_text(text)

    for chunk_id, chunk in enumerate(chunks):

        all_chunks.append({
            "subject": subject,
            "file_name": file_name,
            "chunk_id": chunk_id,
            "chunk_text": chunk
        })

In [23]:
chunks_df = pd.DataFrame(all_chunks)

print(chunks_df.shape)

chunks_df.head()

(4619, 4)


,subject,file_name,chunk_id,chunk_text
0,data_management,0-CommonIntro.pdf,0,data management 2 teaching team 1 course goals...
1,data_management,0-CommonIntro.pdf,1,"d. b. meysman, and mohamed ali. introducing da..."
2,data_management,0-CommonIntro.pdf,2,for the project the project must be approved ...
3,data_management,0-CommonIntro.pdf,3,you must present a new project spotywhy social...
4,data_management,1-DataLifeCycle.pdf,0,data lifecycle 2 methodologies 1 tools 2 phase...


In [24]:
print(chunks_df.iloc[0]["chunk_text"])

data management 2 teaching team 1 course goals and organization 2 exam rules 3 experience from the past 4 teaching team  data management  prof. andrea maurino (lead professor) andrea.maurinounimib.it schedule  see the calendar  november 25 and 26 there will be recorded lecturs  in-presence and registered lessons course organization  main topic: data life cycle  data science not only big data data management data management data visualization data visualization storytelling - machine learning and decision models - statistical modelling what a real data scientist do the truth.  davy cielen, arno d. b. meysman, and mohamed ali. introducing data science, manning, 2016  harrison next generation


In [25]:
chunks_df["chunk_length"] = chunks_df["chunk_text"].apply(len)

chunks_df["chunk_length"].describe()

count    4619.000000
mean      687.038969
std        56.591106
min       103.000000
25%       693.000000
50%       696.000000
75%       698.000000
max       700.000000
Name: chunk_length, dtype: float64

In [26]:
OUTPUT_FOLDER = r"D:\1 Univesrity work\Lect 2\Data semantics\Knowledge-Graph-Enhanced-RAG-System-for-Academic-Question-Answering-in-a-Data-Science-Curriculum\processed"

output_file = os.path.join(OUTPUT_FOLDER, "chunked_data.csv")

chunks_df.to_csv(output_file, index=False)

print("Chunked dataset saved!")

Chunked dataset saved!


In [27]:
def retrieve_chunks(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks_df.iloc[idx]["chunk_text"])  # IMPORTANT FIX

    return results

In [28]:
def build_context(query):

    retrieved_chunks = retrieve_chunks(query)

    kg_facts = retrieve_kg_facts(query)

    context = "\n\n".join(retrieved_chunks[:3])

    if kg_facts:
        context += "\n\nKnowledge Graph Facts:\n"
        context += "\n".join(kg_facts)

    return context

In [29]:
chunks_df["chunk_text"]

0       data management 2 teaching team 1 course goals...
1       d. b. meysman, and mohamed ali. introducing da...
2       for the project  the project must be approved ...
3       you must present a new project spotywhy social...
4       data lifecycle 2 methodologies 1 tools 2 phase...
                              ...                        
4614    class in boosting. in 7th european conference ...
4615    beats over-sampling. in proceedings of the icm...
4616    conference on machine learning, (pp. 179186). ...
4617    conference on knowledge discovery and data min...
4618    recognition techniques for detection of microc...
Name: chunk_text, Length: 4619, dtype: object